# October 1 diagnostic audit checks
All scores are learning-only; this notebook reads retained artifacts and does not issue, fetch or settle a canonical card.

In [ ]:
from pathlib import Path
import csv, json, hashlib, math
audit = Path(r'C:\Users\danie\Desktop\Sports Research\research\verification\settlement_2026-10-01')
rows = list(csv.DictReader((audit/'diagnostic_contracts.csv').open(encoding='utf-8')))
assert len(rows) == 34
for r in rows:
    p, y = float(r['p_stated_original']), int(r['outcome'])
    assert abs((p-y)**2 - float(r['diagnostic_brier'])) < 1e-12
    assert abs(-math.log(p if y else 1-p)-float(r['diagnostic_logloss'])) < 1e-12
    assert r['performance_eligible'] == 'False'
print('34 exact-row diagnostic scores checked; no eligible rows')


In [ ]:
canon = lambda v: json.dumps(v, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode()
prev = '0'*64
for line in (audit/'historical_learning_revisions.jsonl').read_text(encoding='utf-8').splitlines():
    r=json.loads(line); digest=r.pop('revision_sha256')
    assert r['previous_revision_sha256']==prev
    assert hashlib.sha256(canon(r)).hexdigest()==digest
    assert r['payload']['performance_eligible'] is False
    prev=digest
print('Diagnostic chain valid:', prev)
